In [34]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "lewis2017non")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "rspb20170518supp2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [35]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="lewis2017non"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df.rename(columns={"subject": "participant",
                   'species':'species_original',
                   'sex':'sex_original',
                   'latency (seconds)':'latency_in_seconds',
                   "age":"age_in_years",
                   "retreival session":"retrieval_session"}, inplace=True)

In [36]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')


df.columns = df.columns.str.replace(' ', '_', regex=True)
df['search'].replace(' ', '_', inplace=True, regex=True)
# df['participant'].unique()
df['retrieval_session'].unique()

array([ 1.,  2., nan])

In [37]:
df.dropna(subset=['age_in_years'], inplace=True)

spe_2=[] 
for index, row in df.iterrows():
    if row['condition'] == row['sequence']:
      if row['retrieval_session'] == 1.:
        spe_2.append("1")
      if row['retrieval_session'] == 2.:
        spe_2.append("2")
    if row['condition'] != row['sequence']:
      if row['retrieval_session'] == 1.:
        spe_2.append("3")
      if row['retrieval_session'] == 2.:
        spe_2.append("4")
    else:
        pass
df = df.assign(session=spe_2)

df=df.sort_values(by = ['participant', 'session'])


In [38]:
df['age_in_years'] = df['age_in_years'].astype(int)
df['retrieval_session'] = df['retrieval_session'].astype(int)
df['delay'] = df['delay'].astype(int)
# df['age_in_years'].unique()

In [39]:
df['latency_in_seconds']=df['latency_in_seconds'].astype("Int64")
df['binary_search']=df['binary_search'].astype("Int64")
# df['latency_in_seconds'].unique()

In [40]:

studyID_standardized=df[['study_id','participant','age_in_years','sex','species', 'session',
                          'condition','sequence','retrieval_session',
         'delay', 'food',  'search',
       'binary_search', 'latency_in_seconds' ]]
comp_out_path_stand = os.path.join(out_pathway, 'lewis2017non_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'lewis2017non_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
